In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

# Configure Polars to print everything
pl.Config.set_tbl_rows(30)  # -1 means display all rows
pl.Config.set_tbl_cols(-1)  # -1 means display all columns


In [ ]:
KAGGLE_INPUT_BASE_PATH = "/kaggle/input/m5-forecasting-accuracy"
LOCAL_INPUT_BASE_PATH = "./data"
INPUT_BASE_PATH = LOCAL_INPUT_BASE_PATH

### Load data

In [ ]:
calendar_schema_overrides = {
    "wm_yr_wk": pl.Int16,
    **{date_type: pl.Int16 for date_type in ["wday", "month", "year"]},
    **{f"snap_{state}": pl.Int16 for state in ["CA", "TX", "WI"]}
}
CALENDAR_DATA = pl.read_csv(
    f"{INPUT_BASE_PATH}/calendar.csv",
    try_parse_dates=True,
    schema_overrides=calendar_schema_overrides,
)

sell_price_schema_overrides = {
    "wm_yr_wk": pl.Int16,
    "sell_price": pl.Float16,
}
SELL_PRICES = pl.read_csv(
    f"{INPUT_BASE_PATH}/sell_prices.csv",
    schema_overrides=sell_price_schema_overrides
)

SALES_TRAIN_EVALUATION = pl.read_csv(
    f"{INPUT_BASE_PATH}/sales_train_evaluation.csv",
)
SALES_TRAIN_VALIDATION = pl.read_csv(
    f"{INPUT_BASE_PATH}/sales_train_validation.csv",
)
SAMPLE_SUBMISSION = pl.read_csv(
    f"{INPUT_BASE_PATH}/sample_submission.csv",
)

In [ ]:
raw_train_df = (
    SALES_TRAIN_EVALUATION.unpivot(
        index=["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"],
        variable_name="d",
        value_name="sales",
    )
    .with_columns(pl.col("sales").cast(pl.Int16))
    .join(
        # Initially only joining on a subset of columns
        other=CALENDAR_DATA.select(["date", "wm_yr_wk", "d"]),
        on="d",
        how="left",
    )
    .join(
        other=SELL_PRICES,
        on=["store_id", "item_id", "wm_yr_wk"],
        how="left",
    )
    # Remove columns where sell price is null
    .filter(pl.col("sell_price").is_not_null())
    # Add integer index
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16))
)

In [ ]:
# Check no null prices
assert not raw_train_df["sell_price"].is_null().any()

In [ ]:
# Check no duplicates
assert raw_train_df.select(pl.col("id"), pl.col("d_index")).is_duplicated().sum() == 0

In [ ]:
# Check continuous timeseries
id_series_lengths = (
    raw_train_df
    .group_by("id")
    .agg(
        d_index_min=pl.col("d_index").min(),
        d_index_max=pl.col("d_index").max(),
    )
    .with_columns(min_max_diff=pl.col("d_index_max") - pl.col("d_index_min"))
    .join(
        other=raw_train_df.group_by("id").len(name="series_length"),
        on="id",
        how="inner",
        validate="1:1"
    )
)

assert id_series_lengths["min_max_diff"].eq(id_series_lengths["series_length"] - 1).all()

### Lagged features

In [ ]:
SALES_LAGS = [1, 2, 3, 4, 7, 14]

SALES_LAG_DF = (
    raw_train_df
    .select(["id", "d_index", "sales"])
    .with_columns(
        [
            pl.col("sales").shift(lag).over("id", order_by="d_index").alias(f"sales_lagged_{lag}")
            for lag in SALES_LAGS
        ]
    )
    .sort(by=["id", "d_index"])
)

SALES_LAG_DF.head()

### Rolling features

In [ ]:
ROLLING_WINDOWS = [2, 3, 4, 7, 14]

ROLLING_MEAN_DF = (
    raw_train_df
    .select(["id", "d_index", "sales"])
    .with_columns(
        [
            pl.col("sales")
            .shift(1)
            .rolling_mean(window_size=window, min_samples=1)
            .over("id", order_by="d_index")
            .alias(f"sales_rolling_mean_{window}")
            .cast(pl.Float16)
            for window in ROLLING_WINDOWS
        ]
    )
)

ROLLING_MEAN_DF.head()

### Lagged price features

#### TODO

### Rolling price features

#### TODO

### Event features

In [ ]:
N_EVENTS = 2

EVENT_FEATURES_DF = (
    CALENDAR_DATA
    .select(["wm_yr_wk", "d", "date", "event_name_1", "event_type_1", "event_name_2", "event_type_2"])
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16))
    .with_columns(
        [
            pl.when(pl.col(f"event_name_{i}").is_not_null())
            .then(pl.col("d_index"))
            .otherwise(None)
            .alias(f"event_index_{i}")
            for i in (1, 2)
        ]
    )
    # Days until next / since last event
    .with_columns(
        [
            # Days since last event
            *[
                (pl.col("d_index") - pl.col(f"event_index_{i}").forward_fill())
                .alias(f"days_since_event_{i}")
                for i in (1, 2)
            ],
            # Last event type
            *[
                (pl.col(f"event_type_{i}").forward_fill()
                .alias(f"last_event_type_{i}"))
                for i in range(1, N_EVENTS + 1)
            ],
            # Days until next event
            *[
                (pl.col(f"event_index_{i}").backward_fill() - pl.col("d_index"))
                .alias(f"days_until_event_{i}")
                for i in (1, 2)
            ],
            # Next event type
            *[
                (pl.col(f"event_type_{i}").backward_fill()
                .alias(f"next_event_type_{i}"))
                for i in range(1, N_EVENTS + 1)
            ],
            
        ],
    )
    .with_columns(
        [
            # Clip days since last event and event type to 7 days.
            *[
                pl.when(pl.col(f"days_since_event_{i}") <= 7)
                .then(pl.col(f"days_since_event_{i}"))
                .otherwise(None)
                for i in range(1, N_EVENTS + 1)
            ],
            *[
                pl.when(pl.col(f"days_since_event_{i}") <= 7).
                then(pl.col(f"last_event_type_{i}"))
                .otherwise(None)
                for i in range(1, N_EVENTS + 1)
            ],
            # Clip days until next event and event type to 7 days.
            *[              
                pl.when(pl.col(f"days_until_event_{i}") <= 7)
                .then(pl.col(f"days_until_event_{i}"))
                .otherwise(None)
                for i in range(1, N_EVENTS + 1)
            ],
            *[
                pl.when(pl.col(f"days_until_event_{i}") <= 7)
                .then(pl.col(f"next_event_type_{i}"))
                .otherwise(None)
                for i in range(1, N_EVENTS + 1)
            ]
        ]
    )
)

EVENT_FEATURES_DF.head()

### Calendar features

In [ ]:
CALENDAR_FEATURES_DF = (
    CALENDAR_DATA
    .select(["wm_yr_wk", "d", "date", "wday", "month", "year"])
    .with_columns(
        sin_wday=(2 * np.pi * pl.col("wday") / 7).sin().cast(pl.Float16),
        cos_wday=(2 * np.pi * pl.col("wday") / 7).cos().cast(pl.Float16),
        sin_month=(2 * np.pi * pl.col("month") / 12).sin().cast(pl.Float16),
        cos_month=(2 * np.pi * pl.col("month") / 12).cos().cast(pl.Float16),
        is_weekend=(pl.col("wday") >= 6).cast(pl.Float16),
    )
)

CALENDAR_FEATURES_DF.head()

## MERGE ALL TRAIN FEATURES

In [ ]:
FULL_TRAIN_DF = (
    raw_train_df
    .join(
        other=SALES_LAG_DF.select(pl.all().exclude("sales")),
        on=["id", "d_index"],
        how="inner",
        validate="1:1"
    )
    .join(
        other=ROLLING_MEAN_DF.select(pl.all().exclude("sales")),
        on=["id", "d_index"],
        how="inner",
        validate="1:1"
    )
    .join(
        other=(
            CALENDAR_FEATURES_DF
            .join(
                other=EVENT_FEATURES_DF,
                on=["wm_yr_wk", "d", "date"],
                how="inner",
                validate="1:1",
            )
        ),
        on=["wm_yr_wk", "d", "date"],
        how="inner",
        validate="m:1"
    )
)

In [ ]:
FULL_TRAIN_DF.head()

## MERGE ALL SUBMISSION FEATURES

In [ ]:
FINAL_TRAIN_DF_SLICE = (
    FULL_TRAIN_DF.select(["id", "d_index", "sales"])
    .sort(by=["id", "d_index"])
    .group_by("id")
    .tail(14)
)

In [ ]:
_SERIES_ID_RE = (
    r"^(?P<cat>[A-Z]+)_(?P<dept>\d+)_(?P<item>\d+)_(?P<state>[A-Z]+)_(?P<store>\d+)"
    r"_(?:validation|evaluation)$"
)

FORECAST_HORIZON = 28
MAX_TRAIN_INDEX = 1941
FORECAST_IDS = (
    SAMPLE_SUBMISSION
    .filter(SAMPLE_SUBMISSION["id"].str.ends_with("_evaluation"))
    ["id"]
    .unique()
    .to_list()
)

previous_sales_forecast_dfs: list[pl.DataFrame] = []

for f_index in range(1, FORECAST_HORIZON + 1):
    d_index = MAX_TRAIN_INDEX + f_index
    
    # Construct test_df with identifier columns
    test_df = (
        pl.DataFrame({"id": FORECAST_IDS})
        .with_columns(parsed=pl.col("id").str.extract_groups(_SERIES_ID_RE))
        .with_columns(
            item_id=pl.format(
                "{}_{}_{}",
                pl.col("parsed").struct.field("cat"),
                pl.col("parsed").struct.field("dept"),
                pl.col("parsed").struct.field("item"),
            ),
            dept_id=pl.format(
                "{}_{}",
                pl.col("parsed").struct.field("cat"),
                pl.col("parsed").struct.field("dept"),
            ),
            cat_id=pl.format(
                "{}",
                pl.col("parsed").struct.field("cat"),
            ),
            store_id=pl.format(
                "{}_{}",
                pl.col("parsed").struct.field("state"),
                pl.col("parsed").struct.field("store"),
            ),
            state_id=pl.format(
                "{}",
                pl.col("parsed").struct.field("state"),
            )
        )
        .drop("parsed")
        .with_columns(
            d=pl.format("d_{}", d_index),
            d_index=pl.lit(d_index).cast(pl.Int16),
            sales=pl.lit(None).cast(pl.Int16)
        )
    )

    # Construct lagged sales features
    test_sales_lag_df = (
        pl.concat(
            [
                FINAL_TRAIN_DF_SLICE,
                # TODO: Include previous forecasts here
                test_df.select(["id", "d_index", "sales"]),
                
            ]
        )
        .sort(by=["id", "d_index"])
        .with_columns(
            [
                pl.col("sales").shift(lag).over("id", order_by="d_index").alias(f"sales_lagged_{lag}")
                for lag in SALES_LAGS
            ]
        )
    )

    # Construct rolling window features
    test_rolling_window_df = (
        pl.concat(
            [
                FINAL_TRAIN_DF_SLICE,
                # TODO: Include previous forecasts here
                test_df.select(["id", "d_index", "sales"]),
                
            ]
        )
        .sort(by=["id", "d_index"])
        .with_columns(
            [
                pl.col("sales")
                .shift(1)
                .rolling_mean(window_size=window, min_samples=1)
                .over("id", order_by="d_index")
                .alias(f"sales_rolling_mean_{window}")
                .cast(pl.Float16)
                for window in ROLLING_WINDOWS
            ]
        )
    )

    # Join id, lagged, rolling, price, and calendar dfs
    test_df = (
        test_df.drop("sales")
        .join(
            other=test_sales_lag_df.drop("sales"),
            on=["id", "d_index"],
            how="inner",
            validate="1:1",
        )
        .join(
            other=test_rolling_window_df.drop("sales"),
            on=["id", "d_index"],
            how="inner",
            validate="1:1",
        )
        .join(
            other=(
                CALENDAR_FEATURES_DF
                .join(
                    other=EVENT_FEATURES_DF,
                    on=["wm_yr_wk", "d", "date"],
                    how="inner",
                    validate="1:1",
                )
            ),
            on="d",
            how="inner",
            validate="m:1"
        )
        .join(
            other=SELL_PRICES,
            on=["item_id", "store_id", "wm_yr_wk"],
            how="inner",
            validate="1:1"
        )
    )

    # Make predictions
    y_test = ...

    # Save predictions
    ...

###